# Multi-objective Bayesian Optimisation

# Imports

In [ ]:
import torch
import numpy as np

# COBRA
from cobra.flux_analysis import pfba
from cobra.exceptions import OptimizationError

# BayesOpt
from botorch.fit import fit_gpytorch_mll
from botorch.utils.transforms import unnormalize, normalize # for normalising media components
# sampler
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.sampling import sample_simplex

# ACQUISITION FUNCTION - for qPAREGO
from botorch.optim.optimize import optimize_acqf_list 
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import GenericMCObjective
from botorch.utils.multi_objective.scalarization import get_chebyshev_scalarization
from botorch.utils.multi_objective.pareto import is_non_dominated

### Helper Functions & Plotting

In [ ]:
# Plotting functions to be used across notebooks
%run HelperFunctions_MOBO_comprehensive.ipynb

# BayesOpt

## Next Candidate

In [ ]:
def  find_next_candidates(
        medium_tensors_normalised_stacked,
        bounds_tensors_stacked, 
        growth_tensors,
        cost_tensors,
        production_tensors = None,
        opt_objective = "growth-cost",
        n_candidates = 5,
        medium_linear_equality_constraints = None,
        medium_linear_inequality_constraints = None,
        medium_nonlinear_inequality_constraints = None,
        ):
    """
    Finds the next medium composition for which to evaluate cost and optimal growth rate
    * initialises botorch model (list of SingleTaskGP) and mll
    * fits model using mll
    * sets up SobolQMCNormalSampler to sample from posterior
    * uses PAREGO as acquisition function
        * computes posterior mean
        * initialises list of acquisition functions (one per candidate of batch)
        * for each candidate (batch size)
            * uses chebyshev_scalarization to create a vector representation of the chosen objectives
            * defines qLogExpectedImprovement acquisition function and appends to list
        * finds all candidates depending on the acquisition functions

    PARAMETERS
    * medium_tensors_normalised_stacked - tensor - all medium compositions previously evaluated 0-1 normalised, 
    stored as tensors (in order)
    * bounds_tensors_stacked - tensor - encodes the bounds for the medium components 
    * growth_tensors - tensor - corresponding growth rates
    * cost_tensors - tensor - corresponding medium costs
    * production_tensors - tensor - corresponding production rates
    * opt-objective - string - the (multi-)objective for which to find the optimal medium composition
    * n_candidates - integer - how many candidates to find at once
    * medium_linear_equality_constraints -  (list[tuple[Tensor, Tensor, float]] | None) -
        linear equality constaints on medium composition based on formulation constraints
        A list of tuples (indices, coefficients, rhs), with each tuple encoding an equality constraint - 
        has to satisfy the requirements for botorch.optim.optimize.optimize_acqf_list
    * medium_linear_inequality_constraints - (list[tuple[Callable, bool]] | None) -
        linear non-equality constaints on medium composition based on formulation constraints
        has to satisfy the requirements for botorch.optim.optimize.optimize_acqf_list
    * medium_nonliner_inequality_constraints - (list[tuple[Callable, bool]] | None) - 
        A list of tuples representing the nonlinear inequality constraints; 
        has to satisfy the requirements for botorch.optim.optimize.optimize_acqf_list

    RETURNS
    * candidates - tensor - a tensors with n_candidates 0-1 normalised medium compositions to be tested
    """

    '''parameters and conversion to tensors'''
    MC_SAMPLES = 256 #256 # Number of Monte Carlo samples in SobolQMCNormalSampler
    # large values -> slower but possibly better accuracy
    NUM_RESTARTS =  5 #10 # Number of restarts for acquisition function optimisation
    RAW_SAMPLES = 512 # 1024 # Number of raw samples for initialisation of acquisition optimisation

    '''0-1 normalisation of medium composition bounds; ensure that fixed components have bounds [0, 0]'''
    standard_bounds = [] # initialise standard bounds for medium composition
    # check if lower bound and upper bound are the same
    for i in range(len(bounds_tensors_stacked[0])):
        if bounds_tensors_stacked[0][i] == bounds_tensors_stacked[1][i]:
            standard_bounds.append((0.0, 0.0))
        else:
            standard_bounds.append((0.0, 1.0))
    standard_bounds_tensor = torch.tensor(standard_bounds).to(**tkwargs) # normalised bounds for medium composition
    standard_bounds_tensor_stacked = torch.stack([standard_bounds_tensor[:, 0],
                                                  standard_bounds_tensor[:, 1]], dim = 0)

    '''finding the new candidate'''
    # initialise GP model and marginal likelihood (mll)
    mll, model = initialise_model(
        medium_tensors_normalised_stacked,
        growth_tensors,
        opt_objective, 
        cost_tensors, 
        production_tensors)
    
    fit_gpytorch_mll(mll) # Fit the model using the maximum marginal likelihood

    # Compute the posterior mean for the given medium_tensors_stacked using the model
    with torch.no_grad():
        posterior = model.posterior(medium_tensors_normalised_stacked).mean


    # Set up a Sobol quasi-Monte Carlo sampler for sampling from the posterior
    # The sample_shape should correspond to the shape of the posterior samples needed
    # https://botorch.readthedocs.io/en/latest/sampling.html#botorch.sampling.normal.SobolQMCNormalSampler
    sampler = SobolQMCNormalSampler(sample_shape = torch.Size([MC_SAMPLES]), seed = MC_SAMPLES)

    acq_fun_list = [] # List to hold acquisition functions for each candidate
    # Loop to generate acquisition functions for each candidate
    for _ in range(n_candidates):
        # Sample weights from the simplex for Chebyshev scalarization
        weights = sample_simplex(2, **tkwargs).squeeze() # using 2 weights for scalarization (growth and cost or production)

        # Compute the scalarised objective values for all the training points
        # Sample weights from the simplex for Chebyshev scalarization
        if opt_objective == "growth-cost":
            scalarized_objective_values = (
                weights[0] * growth_tensors + 
                weights[1] * cost_tensors)
        elif opt_objective == "growth-production":
            scalarized_objective_values = (
                weights[0] * growth_tensors + 
                weights[1] * production_tensors)
        elif opt_objective == "production-cost":
            scalarized_objective_values = (
                weights[0] * production_tensors + 
                weights[1] * cost_tensors)
        elif opt_objective == "growth-production-cost":
            # using 3 weights for scalarization (growth, production, and cost)
            weights = sample_simplex(3, **tkwargs).squeeze()
            scalarized_objective_values = (
                weights[0] * growth_tensors + 
                weights[1] * production_tensors +
                weights[2] * cost_tensors)

        # Find the best observed scalarized objective value
        best_f = scalarized_objective_values.max().item()

        # Define objective
        objective = GenericMCObjective(
            get_chebyshev_scalarization(weights, posterior)
        )

        # Define the acquisition function using quasi Monte Carlo EI
        acq_fun = qLogExpectedImprovement(
            model = model, # List of SingleTastk GP
            best_f = best_f, # best objective value observed so far - replaces X_baseline in Noisy version
            sampler = sampler, # SobolQMCNormalSampler
            objective = objective, # combination of objectives - Chebyshev scalarization
        )
        acq_fun_list.append(acq_fun)


    # Get new candidates to test
    candidates, _ = optimize_acqf_list(
        acq_function_list = acq_fun_list,  # List of acquisition functions to optimise
        bounds = standard_bounds_tensor_stacked, # The normalised bounds for optimisation
        num_restarts = NUM_RESTARTS, # Number of restarts for optimisation
        raw_samples = RAW_SAMPLES, # Number of raw samples for initialisation (?)
        equality_constraints = medium_linear_equality_constraints,
        inequality_constraints = medium_linear_inequality_constraints,
        nonlinear_inequality_constraints = medium_nonlinear_inequality_constraints,
        options = {"batch_limit": 10, "maxiter": 200,} # Options for acquisition function optimisation
    )
    
    return candidates # candidate_tensor_normalised

## Main

In [ ]:
def media_BayesOpt(
        MetModel, 
        medium = None, 
        bounds = None, 
        costs = None,
        opt_objective = "growth-cost",
        biomass_objective = None,
        production_objective = None,
        n_start = 5,
        data_start = "previous_results.json" or None,
        n_iter = 50,
        n_candidates = 5,
        model_objective = None,
        start_time = None,
        medium_linear_equality_constraints = None,
        medium_linear_inequality_constraints = None,
        medium_nonlinear_inequality_constraints = None,
        use_pfba = False
        ):
    
    """
    Performs medium optimisation for various objectives: trade-off between 
    * growth rate and medium cost
    * growth rate and production rate
    * production rate and medium cost
    * growth rate, production rate and medium cost

    1. Sets default values for medium, bounds and costs if not provided by the user
    2. Performs optimisation n_iter (default = 50) times
        1. calls generate_initial_data(args) to generate initial data points
        2. finds new candidate medium calling find_next_candidate(args)
        3. evaluates new medium for chosen objectives (using either FBA or pFBA)
        4. keeps all values
    3. returns all tested compositions alongside corresponding performance data

    PARAMETERS:
    * MetModel - COBRApy model - the metabolic model to be evaluated
    * medium - dictionary - the medium composition of that model; if not provided defaults to default medium provided by CobraPy
    * bounds - dictionary - upper and lower bounds for the values the medium components are allowed to take,
    determines the search space; if not provided defaults to 0, and current medium value
    * costs - dictionary - the (monetary) cost of each component; if not provided defaults to unit costs
    * opt_objective - string - indicates what is to be optimised
    * biomass_objective - string - the name of the biomass reaction of the chosen model
    * production_objective - string - the name of the producing reaction to be maximised of the chosen model
    * n_start - integer - how many random media compositions are to be created to set up the BayesOpt
    * data_start - string - path to previous results json file to start from previous data; if None, starts from random data using n_start
    * n_iter - integer - how many candidate medium compositions should be found and evaluated
    * n_candidates - integer - how many candidates to find at once
    * model_objective - COBRApy objective - what is set as the objective of the modelled organisms, used in FBA
    * start_time - time object - contains the time when media_BayesOpt was started
    * medium_linear_equality_constraints -  (list[tuple[Tensor, Tensor, float]] | None) -
        linear equality constaints on medium composition based on formulation constraints
        A list of tuples (indices, coefficients, rhs), with each tuple encoding an equality constraint - 
        has to satisfy the requirements for botorch.optim.optimize.optimize_acqf_list
    * medium_linear_inequality_constraints - (list[tuple[Callable, bool]] | None) -
        linear non-equality constaints on medium composition based on formulation constraints
        has to satisfy the requirements for botorch.optim.optimize.optimize_acqf_list
    * medium_nonliner_inequality_constraints - (list[tuple[Callable, bool]] | None) - 
        A list of tuples representing the nonlinear inequality constraints; 
        has to satisfy the requirements for botorch.optim.optimize.optimize_acqf_list
    * use_pfba- Boolean - False -> use standard FBA, True -> use pFBA

    RETURNS:
    A dictionary containing
    * "medium list" - a list of all evaluated medium compositions
    * "medium component bounds" - a dictionary with the upper and lower bounds of each medium components [mmol gDW^-1 h^-1]
    * "medium component costs" - a dictionary with the cost of each medium component [£/mol]
    * "growth rate tensors" - a tensor with corresponding growth rates [1/h]
    * "cost tensors" - a tensor with corresponding total medium costs [mmol gDW^-1 h^-1]
    * "production tensors" - a tensor with corresponding production rates [10^{-3}£ gDW^-1 h^-1]
    * "is pareto" -
    * "optimisation objective" - the objective with which the algorithm was run
    * "biomass objective" - the biomass function to be optimised
    * "production objective" - the production flux to be optimised
    * "model objective" - the COBRApy objective of the model that was used
    * "n_start" - number of random start points
    * "n_iter" - number of iterations
    * "n_candidates" - batch size
    * "lin equ constraints" - medium_linear_equality_constraints
    * "lin inequ constraints" - medium_linear_inequality_constraints
    * "nonlin inequ constraints" - medium_nonlinear_inequality_constraints
    * "pfba" - use_pfba
    """

    '''TEST VALIDITY OF ARGUMENTS'''
    # opt_objective
    valid_opt_objective = {"growth-cost", "growth-production", "production-cost", "growth-production-cost"}
    if opt_objective not in valid_opt_objective:
        raise ValueError(f"opt_objective must be one of {valid_opt_objective}, but got '{opt_objective}'")
    # verify that biomass_objective is not none for otpimisation including growth
    growth_containing_objectives = {"growth-cost", "growth-production", "growth-production-cost"}
    if opt_objective in growth_containing_objectives and biomass_objective == None:
        raise ValueError("Growth is part of the optimisation, please pass a biomass_objective")
    # verify that production_objective is not none for otpimisation including production
    production_containing_objectives = {"growth-production", "production-cost", "growth-production-cost"}
    if opt_objective in production_containing_objectives and production_objective == None:
        raise ValueError("Production is part of the optimisation, please pass a production_objective")


    '''INITIALISE'''
    # Set default values for medium, boundaries and costs
    if medium is None:
        medium = MetModel.medium  # Default medium to model.medium if not provided
    if bounds is None:
        # if no bounds are provided, set the lower limit to 0 and upper to the value in medium
        bounds = {key: (0, medium[key]) for key in medium.keys()}
    if costs is None:
        # set unit costs if no costs are provided
        costs = {key: 1 for key in medium.keys()}

    # if a model_objective is given, set it 
    if model_objective:
        MetModel.objective = model_objective
    elif model_objective is None:
        model_objective = MetModel.objective
    
    '''TEST THAT MEDIUM, BOUNDS, AND COST HAVE THE SAME ENTRIES'''
    same_bounds = (medium.keys() == bounds.keys()) and (medium.keys() == costs.keys())
    if not same_bounds:
        raise ValueError("medium, bounds, and costs must have the exact same keys")
    
    if n_start is not None:
        '''GET RANDOM INITIAL DATA POINTS'''
        # generate n_start initial data points (parameters and corresponding cost + growth rate)
        initial_para, initial_growth, initial_production, initial_cost = generate_initial_data(
            MetModel, medium, bounds, costs,
            n_samples = n_start, 
            opt_objective = opt_objective, 
            biomass_objective = biomass_objective, 
            production_objective = production_objective,
            use_pfba = use_pfba)
        
        medium_list = initial_para # list of dictonaries
        medium_keys = medium_list[-1].keys() # extract keys from medium_list
        growth_tensors = initial_growth
        #growth_tensors_normalised = normalise_1Dtensors(growth_tensors)
        production_tensors = initial_production
        production_tensors_normalised = normalise_1Dtensors(production_tensors)
        cost_tensors = initial_cost # tensor
        cost_tensors_normalised = normalise_1Dtensors(cost_tensors) # min-max normalised
        is_pareto = []
    
    elif data_start is not None:
        '''LOAD PREVIOUS DATA'''
        previous_results = JSON_deserialize_load_results(data_start, MetModel)
        # select pareto front only
        previous_results_df = results_dic_to_df(previous_results)
        pareto_previous_results_df = previous_results_df.loc[previous_results_df['is pareto'] == True]
        # convert pareto front points to dictionary
        dic_pareto_previous_results = results_df_to_dic(pareto_previous_results_df)
        # identify fixed components (not stored in previous results file)
        fixed_components = {}
        for component, bound in bounds.items():
            if bound[0] == bound[1]:
                fixed_components[component] = bound[0]
        # add fixed components to each medium composition
        for medium in dic_pareto_previous_results["medium list"]:
            for component, value in fixed_components.items():
                medium[component] = value
        # extract relevant data
        medium_list = dic_pareto_previous_results["medium list"] # list of dictonaries
        medium_keys = medium_list[-1].keys() # extract keys from medium_list
        growth_tensors = dic_pareto_previous_results["growth rate tensors"]
        production_tensors = dic_pareto_previous_results["production tensors"]
        production_tensors_normalised = normalise_1Dtensors(production_tensors)
        cost_tensors = dic_pareto_previous_results["cost tensors"] # tensor
        cost_tensors_normalised = normalise_1Dtensors(cost_tensors) # min-max normalised
        is_pareto = dic_pareto_previous_results["is pareto"]
    
    '''CONVERT MEDIUM_LIST TO TENSOR'''
    # convert bounds from dictionary to tensor
    bounds_tensor = torch.tensor(list(bounds.values()), dtype=torch.double).to(**tkwargs) # [x, 2]
    # Stack the lower and upper bounds to match the expected format
    bounds_tensors_stacked = torch.stack([bounds_tensor[:, 0], bounds_tensor[:, 1]], dim=0)

    # normalise medium composition
    medium_tensors_normalised = [] # initialise empty list
    for m in range(len(medium_list)):
        # transform current medium to tensor
        medium_m = medium_list[m]
        medium_m_tensor = torch.tensor(list(medium_m.values()), dtype=torch.double).to(**tkwargs) # [x]
        # normalise medium composition using the bounds
        normalised_medium_m = normalize(medium_m_tensor, bounds_tensors_stacked)
        # Append the normalized tensor to the list
        medium_tensors_normalised.append(normalised_medium_m)

    '''MAIN LOOP'''
    for i in range(n_iter):
        # Stack the list of tensors along a new dimension (dim=0) -> single tensor
        medium_tensors_normalised_stacked = torch.stack(medium_tensors_normalised, dim = 0) # normalised
        # Use BayesOpt to change medium

        '''Need to pass normalised cost and production'''        
        candidates_tensor_normalised = find_next_candidates(
            medium_tensors_normalised_stacked,
            bounds_tensors_stacked,
            growth_tensors = growth_tensors,
            cost_tensors = (1 - cost_tensors_normalised), # because costs should be minimised but function maximises
            production_tensors = production_tensors_normalised,
            opt_objective = opt_objective,
            n_candidates = n_candidates,
            medium_linear_equality_constraints = medium_linear_equality_constraints,
            medium_linear_inequality_constraints = medium_linear_inequality_constraints,
            medium_nonlinear_inequality_constraints = medium_nonlinear_inequality_constraints,
            )

        # if n_candidates > 1, each candidate needs to be evaluated individually
        for candidate_tensor_normalised in candidates_tensor_normalised:
                
            # unnormlise new candidate
            candidate_tensor_unnormalised = unnormalize(candidate_tensor_normalised, bounds_tensors_stacked)
            # convert back to dictionary            
            candidate_medium = convert_to_dict(candidate_tensor_unnormalised, medium_keys)
                
            # for new medium compute new values
            cost_tot = calc_cost_tot(costs, candidate_medium) # tensor
            MetModel.medium = candidate_medium # reassign medium
            
            # perform FBA
            fba_solution = MetModel.optimize()

            # if use_pfba == False, proceed with standard FBA
            if use_pfba == False:
                # extract growth rate
                if biomass_objective is None:
                    FBA_growth = -1
                else:
                    FBA_growth = fba_solution.fluxes[biomass_objective]
                # some model compositions lead to FBA returns NaN or negative numbers
                # to avoid them from breaking the algorithm, set growth to zero
                if (np.isnan(FBA_growth) or FBA_growth < 0):
                    FBA_growth = 0

                if opt_objective == "growth-cost":
                    FBA_production = -1
                    
                elif (opt_objective == "growth-production" or 
                    opt_objective == "production-cost" or
                    opt_objective == "growth-production-cost"):
                    FBA_production = fba_solution.fluxes[production_objective]
                    if (np.isnan(FBA_production) or FBA_production < 0):
                        FBA_production = 0
            
            # if use_pfba == True, proceed with pFBA
            elif use_pfba == True:
                # catch infeasible, unbound, time-limited or iteration-limited solutions
                if fba_solution.status != "optimal":
                    if biomass_objective is None:
                        FBA_growth = -1
                    else:
                        # penalise
                        FBA_growth = 0

                    if opt_objective == "growth-cost":
                        FBA_production = -1
                        
                    elif (opt_objective == "growth-production" or 
                        opt_objective == "production-cost" or
                        opt_objective == "growth-production-cost"):
                        # penalise                  
                        FBA_production = 0
                
                # when the medium composition leads to a feasible problem - run pFBA
                else:
                    try:
                        pfba_solution = pfba(MetModel)
                        # extract growth rate
                        if biomass_objective is None:
                            FBA_growth = -1
                        else:
                            FBA_growth = pfba_solution.fluxes[biomass_objective]
                        # some model compositions lead to FBA returns NaN or negative numbers
                        # to avoid them from breaking the algorithm, set growth to zero
                        if (np.isnan(FBA_growth) or FBA_growth < 0):
                            FBA_growth = 0

                        if opt_objective == "growth-cost":
                            FBA_production = -1
                            
                        elif (opt_objective == "growth-production" or 
                            opt_objective == "production-cost" or
                            opt_objective == "growth-production-cost"):
                            FBA_production = pfba_solution.fluxes[production_objective]
                            if (np.isnan(FBA_production) or FBA_production < 0):
                                FBA_production = 0
                    
                    except OptimizationError:
                        FBA_growth = 0
                        if opt_objective == "growth-cost":
                            FBA_production = -1
                        else:
                            FBA_production = 0


            '''APPEND RESULTS TO TENSORS AND NORMALISE'''
            # medium lists
            medium_list.append(candidate_medium)
            medium_tensors_normalised.append(candidate_tensor_normalised)
            # growth
            FBA_growth_tensor = torch.tensor([FBA_growth], dtype=torch.double).to(**tkwargs)
            growth_tensors = torch.cat((growth_tensors, FBA_growth_tensor), dim = 0)  # Concatenate along dimension 0
            #growth_tensors_normalised = normalise_1Dtensors(growth_tensors)
            # production
            FBA_production_tensor = torch.tensor([FBA_production], dtype = torch.double).to(**tkwargs)
            production_tensors = torch.cat((production_tensors, FBA_production_tensor), dim = 0)
            production_tensors_normalised = normalise_1Dtensors(production_tensors)
            # cost
            cost_tensors = torch.cat((cost_tensors, cost_tot), dim = 0)  # Concatenate along dimension 0 (1D tensors)
            cost_tensors_normalised = normalise_1Dtensors(cost_tensors) # new min-max normalisation

        # Print Update about how far the programme has progressed in what time.
        if ((i+1)%10 == 0):
            print("Iteration:\t", i+1, print_runtime(start_time))

    '''FIND POINTS ON PARETO FRONT'''
    # Find all points on pareto front and return them     
    # Stack all (two/three) objectives into a single 2D tensor
    # rows: candidates; columns: objectives
    # is_non_dominated assumes maximisation -> negate costs
    if opt_objective == "growth-cost":
        y = torch.stack((growth_tensors, cost_tensors*(-1)), dim = 1)
    elif opt_objective == "growth-production":
        y = torch.stack((growth_tensors, production_tensors), dim = 1)
    elif opt_objective == "production-cost":
        y = torch.stack((production_tensors, cost_tensors*(-1)), dim = 1)
    elif opt_objective == "growth-production-cost":
        y = torch.stack((growth_tensors, production_tensors, cost_tensors*(-1)), dim = 1)

    # Compute non-dominated (Pareto front) points; i.e. optimal trade.offs
    is_pareto = is_non_dominated((y).to(**tkwargs))
    
    return {
        "medium list" : medium_list, 
        "medium component bounds" : bounds, # [mmol gDW^-1 h^-1]
        "medium component costs" : costs, # [£/mol]
        "growth rate tensors" : growth_tensors, # [1/h]
        "production tensors" : production_tensors, # [mmol gDW^-1 h^-1]
        "cost tensors" : cost_tensors, # [10^{-3}£ gDW^-1 h^-1]
        "is pareto" : is_pareto, 
        "optimisation objective" : opt_objective,
        "biomass objective" : biomass_objective,
        "production objective" : production_objective,
        "model objective" : model_objective,
        "n_start" : n_start,
        "n_iter" : n_iter,
        "n_candidates" : n_candidates,
        "lin equ constraints" : medium_linear_equality_constraints,
        "lin inequ constraints" : medium_linear_inequality_constraints,
        "nonlin inequ constraints" : medium_nonlinear_inequality_constraints,
        "pFBA" : use_pfba
        }
